# DenseNet-121 전체 PA 학습: CSV 생성 → Training Job 제출

**전체 흐름** (위에서부터 순서대로 실행하고 자면 됨):
1. 소스 CSV 3종 다운로드 (metadata, split, chexpert)
2. 전체 PA Master CSV 생성 (~94,000장)
3. CSV + pos_weight S3 업로드
4. train.py 패키징 + S3 업로드
5. SageMaker Training Job 제출 (스팟, ~7-8시간)

> Training Job은 제출 후 독립 실행됨. 노트북 꺼도 됨.

**vs p10 학습**: p10(9,118장) → 전체 PA(~94,000장). 10배 데이터로 실전 모델 생성.

## Part 1: 환경 설정

In [ ]:
import os, time, json, tarfile, subprocess
import numpy as np
import pandas as pd
import boto3

WORK_DIR = '/home/ec2-user/SageMaker/densenet_full_data'
os.makedirs(WORK_DIR, exist_ok=True)

BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
IMAGE_BUCKET = 'say1-pre-project-5'
S3_CSV_PREFIX = 'data/densenet_csv'
S3_CODE_PREFIX = 'code'

sm_client = boto3.client('sagemaker', region_name='ap-northeast-2')
s3_client = boto3.client('s3')

LABEL_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
    'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
    'Pneumothorax', 'Support Devices'
]

print(f"작업 디렉토리: {WORK_DIR}")
print(f"작업 버킷: {BUCKET}")
print(f"이미지 버킷: {IMAGE_BUCKET}")
print("환경 설정 완료")

## Part 2: 소스 CSV 다운로드

MIMIC-CXR 공식 CSV 3종을 S3에서 다운로드합니다.

In [ ]:
CSV_DIR = f's3://{BUCKET}/data/mimic-cxr-csv'

csv_files = [
    'mimic-cxr-2.0.0-metadata.csv',
    'mimic-cxr-2.0.0-split.csv',
    'mimic-cxr-2.0.0-chexpert.csv',
]

for f in csv_files:
    local = os.path.join(WORK_DIR, f)
    if os.path.exists(local):
        print(f"이미 존재: {f}")
    else:
        print(f"다운로드: {f}")
        !aws s3 cp {CSV_DIR}/{f} {local}

print("\nCSV 다운로드 완료")

## Part 3: 전체 PA Master CSV 생성

파이프라인:
1. metadata + chexpert 병합 (study 단위)
2. split 병합 (dicom 단위)
3. PA only 필터링 (377K → ~96K)
4. U-Ones 라벨 변환 (-1 → 1)
5. image_path 생성 (S3 key)
6. pos_weight 계산

In [ ]:
%%time
# CSV 로드
meta = pd.read_csv(os.path.join(WORK_DIR, 'mimic-cxr-2.0.0-metadata.csv'))
split = pd.read_csv(os.path.join(WORK_DIR, 'mimic-cxr-2.0.0-split.csv'))
chex = pd.read_csv(os.path.join(WORK_DIR, 'mimic-cxr-2.0.0-chexpert.csv'))

print(f"metadata: {len(meta):,}행")
print(f"split:    {len(split):,}행")
print(f"chexpert: {len(chex):,}행")

# 1. PA만 필터
meta_pa = meta[meta['ViewPosition'] == 'PA'].copy()
print(f"\nPA 필터: {len(meta):,} → {len(meta_pa):,}")

# 2. chexpert 라벨 병합 (study 단위, inner join — 라벨 없는 study 제외)
df = meta_pa.merge(chex, on=['subject_id', 'study_id'], how='inner')
print(f"라벨 병합: {len(df):,}")

# 3. split 병합
df = df.merge(split, on=['subject_id', 'study_id', 'dicom_id'], how='inner')
print(f"Split 병합: {len(df):,}")

# 4. U-Ones: NaN → 0, -1(uncertain) → 1
for col in LABEL_COLS:
    df[col] = df[col].fillna(0).replace(-1, 1).astype(int)

# 5. image_path 생성 (say1-pre-project-5 내 S3 key)
def build_image_path(row):
    sid = str(row['subject_id'])
    group = sid[:2]
    return f"files/p{group}/p{sid}/s{row['study_id']}/{row['dicom_id']}.jpg"

df['image_path'] = df.apply(build_image_path, axis=1)

print(f"\n최종 데이터: {len(df):,}행")
print(f"\nSplit 분포:")
print(df['split'].value_counts())
print(f"\nimage_path 샘플: {df['image_path'].iloc[0]}")

In [ ]:
# pos_weight 계산 (train split 기준)
train_df = df[df['split'] == 'train']
pos_weights = {}
print(f"pos_weight (train {len(train_df):,}행 기준):")
print(f"{'질환':<35} {'양성':>8} {'음성':>8} {'weight':>10}")
print('─' * 65)

for col in LABEL_COLS:
    pos = int(train_df[col].sum())
    neg = len(train_df) - pos
    w = round(neg / max(pos, 1), 4)
    pos_weights[col] = w
    print(f"{col:<35} {pos:>8,} {neg:>8,} {w:>10.4f}")

# 라벨 분포 확인
print(f"\n라벨 분포 (전체):")
for col in LABEL_COLS:
    pct = df[col].mean() * 100
    print(f"  {col:<35} {pct:>6.2f}%")

In [ ]:
# CSV 저장
output_cols = ['subject_id', 'study_id', 'dicom_id', 'image_path', 'split'] + LABEL_COLS
csv_path = os.path.join(WORK_DIR, 'full_pa_train_ready.csv')
df[output_cols].to_csv(csv_path, index=False)
size_mb = os.path.getsize(csv_path) / (1024**2)
print(f"저장: {csv_path} ({size_mb:.1f} MB)")
print(f"행: {len(df):,}")

# pos_weight JSON 저장
pw_path = os.path.join(WORK_DIR, 'full_pa_pos_weights.json')
with open(pw_path, 'w') as f:
    json.dump(pos_weights, f, indent=2)
print(f"pos_weight 저장: {pw_path}")

## Part 4: S3 업로드 (CSV + pos_weight)

In [ ]:
# CSV + pos_weight S3 업로드
!aws s3 cp {csv_path} s3://{BUCKET}/{S3_CSV_PREFIX}/full_pa_train_ready.csv
!aws s3 cp {pw_path} s3://{BUCKET}/{S3_CSV_PREFIX}/full_pa_pos_weights.json

print("\n업로드 완료:")
!aws s3 ls s3://{BUCKET}/{S3_CSV_PREFIX}/

## Part 5: 학습 코드 패키징 + S3 업로드

`train.py`를 `sourcedir.tar.gz`로 압축하여 S3에 업로드.

In [ ]:
# train.py 찾기
TRAIN_SCRIPT = None
candidates = [
    '/home/ec2-user/SageMaker/forpreproject/layer2_detection/densenet/train.py',
    os.path.join(WORK_DIR, 'train.py'),
]
for path in candidates:
    if os.path.exists(path):
        TRAIN_SCRIPT = path
        break

if TRAIN_SCRIPT is None:
    print("S3에서 train.py 다운로드...")
    TRAIN_SCRIPT = os.path.join(WORK_DIR, 'train.py')
    !aws s3 cp s3://{BUCKET}/code/densenet/train.py {TRAIN_SCRIPT} 2>/dev/null || true
    if not os.path.exists(TRAIN_SCRIPT):
        raise FileNotFoundError("train.py를 찾을 수 없습니다!")

print(f"학습 스크립트: {TRAIN_SCRIPT}")

# tar.gz 생성
tarball_path = os.path.join(WORK_DIR, 'densenet_sourcedir.tar.gz')
with tarfile.open(tarball_path, 'w:gz') as tar:
    tar.add(TRAIN_SCRIPT, arcname='train.py')
print(f"패키징 완료: {tarball_path} ({os.path.getsize(tarball_path)/1024:.1f} KB)")

# S3 업로드
s3_code_key = f'{S3_CODE_PREFIX}/densenet_full_sourcedir.tar.gz'
s3_client.upload_file(tarball_path, BUCKET, s3_code_key)
print(f"업로드: s3://{BUCKET}/{s3_code_key}")

## Part 6: SageMaker Training Job 제출

**이 셀 실행 후 자도 됩니다!**

- 인스턴스: ml.g5.xlarge (A10G 24GB)
- 이미지: say1-pre-project-5에서 PA 이미지만 S3 직접 다운로드 (~28GB, ~10분)
- 학습: Stage1 5에폭(classifier) + Stage2 25에폭(full) = 30에폭
- 예상 소요: ~7-8시간
- 예상 비용: ~$2-3 (스팟)

```python
# 내일 아침에 확인
sm_client.describe_training_job(TrainingJobName='densenet121-full-pa-v1')['TrainingJobStatus']
```

In [ ]:
JOB_NAME = 'densenet121-full-pa-v1'

training_params = {
    "TrainingJobName": JOB_NAME,
    "RoleArn": "arn:aws:iam::666803869796:role/SKKU_SageMaker_Role",
    "AlgorithmSpecification": {
        "TrainingImage": "763104351884.dkr.ecr.ap-northeast-2.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker",
        "TrainingInputMode": "File"
    },
    "HyperParameters": {
        "sagemaker_program": "train.py",
        "sagemaker_submit_directory": f"s3://{BUCKET}/{s3_code_key}",
        "sagemaker_region": "ap-northeast-2",
        "batch-size": "32",
        "stage1-epochs": "5",
        "stage2-epochs": "25",
        "lr": "0.0001",
        "num-workers": "4",
        "image-bucket": IMAGE_BUCKET
    },
    "ResourceConfig": {
        "InstanceType": "ml.g5.xlarge",
        "InstanceCount": 1,
        "VolumeSizeInGB": 100
    },
    "InputDataConfig": [
        {
            "ChannelName": "csv",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{BUCKET}/{S3_CSV_PREFIX}/",
                    "S3DataDistributionType": "FullyReplicated"
                }
            }
        }
    ],
    "CheckpointConfig": {
        "S3Uri": f"s3://{BUCKET}/checkpoints/densenet-full-pa/"
    },
    "OutputDataConfig": {
        "S3OutputPath": f"s3://{BUCKET}/output/"
    },
    "EnableManagedSpotTraining": True,
    "StoppingCondition": {
        "MaxRuntimeInSeconds": 43200,
        "MaxWaitTimeInSeconds": 172800
    },
    "Tags": [
        {"Key": "name", "Value": "say2-preproject-6team-hyunwoo"}
    ]
}

print("=" * 60)
print(f"Training Job 제출: {JOB_NAME}")
print(f"인스턴스: ml.g5.xlarge (GPU A10G 24GB)")
print(f"데이터: 전체 PA {len(df):,}장 (S3 직접 다운로드 ~28GB)")
print(f"학습: Stage1(5ep, classifier) + Stage2(25ep, full) = 30에폭")
print(f"스팟 학습: 활성화 (비용 ~70% 절감)")
print(f"예상 소요: 7~8시간 (다운로드 ~10분 + 학습 ~7시간)")
print(f"예상 비용: ~$2-3")
print("=" * 60)

response = sm_client.create_training_job(**training_params)
print(f"\n제출 완료! ARN: {response['TrainingJobArn']}")
print(f"\n내일 아침에 확인:")
print(f"  sm_client.describe_training_job(TrainingJobName='{JOB_NAME}')['TrainingJobStatus']")

## Part 7: (다음날) 상태 확인 + 결과 다운로드

In [ ]:
# Training Job 상태 확인
JOB_NAME = 'densenet121-full-pa-v1'
desc = sm_client.describe_training_job(TrainingJobName=JOB_NAME)

print(f"Job: {JOB_NAME}")
print(f"Status: {desc['TrainingJobStatus']}")
if 'SecondaryStatus' in desc:
    print(f"Detail: {desc['SecondaryStatus']}")
if desc['TrainingJobStatus'] == 'Completed':
    print(f"\nTraining Time: {desc.get('TrainingTimeInSeconds', 0)/60:.1f}분")
    print(f"Billable Time: {desc.get('BillableTimeInSeconds', 0)/60:.1f}분")
    savings = 1 - desc.get('BillableTimeInSeconds',0) / max(desc.get('TrainingTimeInSeconds',1),1)
    print(f"스팟 절감율: {savings*100:.0f}%")
    print(f"Model: s3://{BUCKET}/output/{JOB_NAME}/output/model.tar.gz")
elif desc['TrainingJobStatus'] == 'Failed':
    print(f"\nFailure Reason: {desc.get('FailureReason', 'unknown')}")

In [ ]:
# 학습 완료 후 결과 다운로드
JOB_NAME = 'densenet121-full-pa-v1'
MODEL_TAR = f'/tmp/{JOB_NAME}_model.tar.gz'

!aws s3 cp s3://{BUCKET}/output/{JOB_NAME}/output/model.tar.gz {MODEL_TAR}

import tarfile
with tarfile.open(MODEL_TAR, 'r:gz') as tar:
    tar.extractall('/tmp/densenet_model/')

# 결과 확인
with open('/tmp/densenet_model/results.json') as f:
    results = json.load(f)

print("=" * 60)
print("DenseNet-121 전체 PA 학습 결과")
print("=" * 60)
print(f"Mean AUROC: {results['mean_auroc']:.4f}")
print(f"Test Loss:  {results['test_loss']:.4f}")
print(f"학습 시간:  {results['training_time_minutes']:.1f}분")
print(f"\n{'질환':<35} {'AUROC':>8}")
print('─' * 45)
for disease, auroc in results['per_class_auroc'].items():
    if auroc is not None:
        print(f"{disease:<35} {auroc:>8.4f}")
    else:
        print(f"{disease:<35} {'N/A':>8}")

# Lambda 배포용 S3 복사
!aws s3 cp /tmp/densenet_model/best_model.pth s3://{BUCKET}/models/densenet121_full_pa.pth
!aws s3 cp /tmp/densenet_model/results.json s3://{BUCKET}/models/densenet121_full_pa_results.json
print(f"\nLambda 배포용 업로드 완료:")
print(f"  s3://{BUCKET}/models/densenet121_full_pa.pth")
print(f"  s3://{BUCKET}/models/densenet121_full_pa_results.json")